# 02 - Chunking and Index Preparation

This notebook reads the cleaned document text, splits it into retrieval friendly chunks,
writes the chunk table, and enables Change Data Feed for Vector Search.

In [0]:
from typing import List

In [0]:
# -----------------------------
# Configuration
# -----------------------------
CATALOG = "workspace"
SCHEMA = "rag_demo"

PARSED_DOCS_TABLE = f"{CATALOG}.{SCHEMA}.parsed_docs_clean"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.rag_chunks"
DOC_TYPE = "runbook_library"

# Chunk size in approximate words
MAX_WORDS_PER_CHUNK = 220
OVERLAP_WORDS = 40

# -----------------------------
# Print
# -----------------------------
print(f"PARSED_DOCS_TABLE: {PARSED_DOCS_TABLE}")
print(f"CHUNKS_TABLE: {CHUNKS_TABLE}")
print(f"DOC_TYPE: {DOC_TYPE}")

## Helper Functions

In [0]:
def split_into_chunks_with_overlap(
    text: str,
    max_words: int = 220,
    overlap_words: int = 40
) -> List[str]:
    """
    Create chunks with overlap for better retrieval.

    Strategy:
    - Split into paragraphs
    - Build chunks up to max_words
    - Add overlap from previous chunk
    """

    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()] #Splits text using double newline
    
    chunks = []
    current_chunk = []
    current_word_count = 0

    for paragraph in paragraphs:
        words = paragraph.split()
        paragraph_word_count = len(words)

        if current_word_count + paragraph_word_count > max_words and current_chunk:
            # finalize chunk
            chunk_text = "\n\n".join(current_chunk)
            chunks.append(chunk_text)

            # create overlap from last chunk
            last_words = chunk_text.split()[-overlap_words:]
            overlap_text = " ".join(last_words)

            current_chunk = [overlap_text, paragraph]
            current_word_count = len(overlap_text.split()) + paragraph_word_count
        else:
            current_chunk.append(paragraph)
            current_word_count += paragraph_word_count

    if current_chunk:
        chunks.append("\n\n".join(current_chunk))

    return chunks

## Read parsed document

In [0]:
parsed_df = spark.table(PARSED_DOCS_TABLE)
display(parsed_df)

## Document Metadata Extraction

In [0]:
row = parsed_df.collect()[0]

file_path = row["file_path"]
file_name = row["file_name"]
clean_text = row["text"]

print(f"File name: {file_name}")
print(f"Document length (characters): {len(clean_text)}")

## Text Chunking with Overlap

In [0]:
chunks = split_into_chunks_with_overlap(
    clean_text,
    max_words=MAX_WORDS_PER_CHUNK,
    overlap_words=OVERLAP_WORDS
)

print(f"Number of chunks created: {len(chunks)}")

## Preview Generated Text Chunks

In [0]:
for i, chunk in enumerate(chunks[:5], start=1):
    print("\n" + "=" * 100)
    print(f"Chunk {i}")
    print("=" * 100)
    print(chunk[:1500])

## Build and save chunk table

In [0]:
chunk_rows = [
    (f"chunk_{i}", file_path, file_name, "runbook_library", chunk)
    for i, chunk in enumerate(chunks, start=1)
]

chunk_df = spark.createDataFrame(
    chunk_rows,
    ["chunk_id", "file_path", "file_name", "doc_type", "chunk"]
)

display(chunk_df)
chunk_df.write.mode("overwrite").saveAsTable(CHUNKS_TABLE)

## Enable Change Data Feed
Required for Delta Sync Vector Search index.

In [0]:
spark.sql(f"""
ALTER TABLE {CHUNKS_TABLE}
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

spark.sql(f"SHOW TBLPROPERTIES {CHUNKS_TABLE}").show(truncate=False)